## ¿Qué vamos a demostrar?

Vamos a construir un archivo CSV sintético de 2 millón de filas y luego haremos cinco experimentos:

1. **Carga ingenua con pandas**  
   Leer el archivo “tal cual” y ver cuánto tarda y cuánta memoria usa.

2. **Carga optimizada**  
   Leer el mismo archivo definiendo tipos de datos más eficientes.

3. **Tipos categóricos**  
   Ver cuánto ahorran las columnas `category` cuando hay muchos valores repetidos.

4. **Chunking**  
   Procesar el archivo por partes sin cargarlo completo a memoria.

5. **Paralelización**  
   Comparar una versión secuencial contra una versión paralela simple.

Al final, vamos a sacar conclusiones prácticas.

## Idea clave antes de empezar

En pandas, la mejor estrategia casi siempre es:

1. **leer menos datos**
2. **usar mejores tipos**
3. **usar operaciones vectorizadas, es decir operaciones a nivel de columna**
4. **usar chunking si hace falta**
5. **paralelizar solo si realmente aporta**

Eso significa que la paralelización normalmente es el último paso, no el primero.

In [0]:

# Se usa para forzar la liberación de memoria RAM limpiando objetos que ya no se usan.
import gc
# Permite interactuar con el sistema operativo
import os
#Proporciona funciones relacionadas con el tiempo, ideal para medir cuánto tarda en ejecutarse un bloque de código
import time
#Manejo de rutas
from pathlib import Path
#Manejo de arreglos
import numpy as np
#Manejo de dataframes = tablas
import pandas as pd
#Se utiliza para crear gráficos y visualizaciones de datos.
import matplotlib.pyplot as plt
#Facilita la ejecución de tareas en paralelo
from joblib import Parallel, delayed
#Sirve para extraer información como de cuanta memoria RAMse esta consumiendo
import psutil

#Obtiene el proceso que se esta ejecutando actualmente
process = psutil.Process(os.getpid())

def rss_mb():
    """
    Calcula la memoria RAM (Resident Set Size) que está consumiendo el proceso actual de Python.

    Returns:
        float: El uso de memoria física actual en Megabytes (MB).
    """
    return process.memory_info().rss / 1048576 # Para pasar de bytes a MB

def df_mem_mb(df):
    """
    Calcula el tamaño total en memoria RAM de un DataFrame de Pandas.

    Args:
        df (pandas.DataFrame): El DataFrame cuyo tamaño se desea medir.

    Returns:
        float: El tamaño real y exacto del DataFrame en Megabytes (MB).
    """
    return df.memory_usage(deep=True).sum() / 1048576

def timer():
    """
    Obtiene una marca de tiempo de alta resolución para medir el rendimiento del código.

    Returns:
        float: El valor actual del contador de rendimiento en fracciones de segundo.
    """
    return time.perf_counter()

# =========================
# CONFIGURA AQUÍ TU VOLUME
# =========================
CATALOG = "workspace"
SCHEMA = "raw"
VOLUME = "ventas"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

CSV_PATH = Path("/Volumes/workspace/raw/ventas") / "ventas.csv"

## 1. Crear el dataset de 2.000.000 de filas

Vamos a simular ventas con columnas típicas de negocio:

- `id_venta`
- `fecha`
- `ciudad`
- `canal`
- `estado`
- `producto`
- `segmento`
- `cantidad`
- `precio_unitario`
- `descuento_pct`
- `monto`

Este dataset está diseñado para que algunas columnas tengan **muchos valores repetidos**, lo cual nos permitirá ver claramente el beneficio de los tipos categóricos.

In [0]:
rng = np.random.default_rng(42) # Motor de generación de números aleatorios

n = 2000000 # número de datos a simular

#Arreglo de ciudades
ciudades = np.array([
    "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga",
    "Pereira", "Cartagena", "Manizales", "Ibagué", "Pasto"
])

#Arreglo de canales
canales = np.array(["Web", "Tienda", "CallCenter", "App", "Aliado"])

#Arreglo de estados
estados = np.array(["APROBADA", "PENDIENTE", "RECHAZADA", "DEVUELTA"])

#Arreglo de productos
productos = np.array(["Hogar", "Tecnología", "Ropa", "Deportes", "Belleza", "Juguetes", "Libros", "Salud"])

#Arreglo de segmentos
segmentos = np.array(["Retail", "Pyme", "VIP"])

#Arreglo de fechas
fechas = np.datetime64("2025-01-01") + rng.integers(0, 365, size=n).astype("timedelta64[D]")

#Construcción del dataframe
df_generado = pd.DataFrame({
    "id_venta": np.arange(1, n + 1, dtype=np.int32),
    "fecha": fechas.astype(str),
    "ciudad": rng.choice(ciudades, size=n), # Todos tienen la misma probabilidad de salir
    "canal": rng.choice(canales, size=n, p=[0.22, 0.28, 0.12, 0.28, 0.10]), # Probabilidades especificas
    "estado": rng.choice(estados, size=n, p=[0.72, 0.10, 0.14, 0.04]),
    "producto": rng.choice(productos, size=n),
    "segmento": rng.choice(segmentos, size=n, p=[0.70, 0.20, 0.10]),
    "cantidad": rng.integers(1, 6, size=n, dtype=np.int8),
    "precio_unitario": rng.uniform(10, 500, size=n).round(2).astype(np.float32),
    "descuento_pct": rng.choice([0, 0.05, 0.10, 0.15, 0.20], size=n, p=[0.35, 0.25, 0.20, 0.15, 0.05]).astype(np.float32),
})

df_generado["monto"] = (
    df_generado["cantidad"] * df_generado["precio_unitario"] * (1 - df_generado["descuento_pct"])
).round(2).astype(np.float32)

t0 = timer()
df_generado.to_csv(CSV_PATH, index=False)
t1 = timer()

file_size_mb = CSV_PATH.stat().st_size / 1048576

print(f"Filas generadas: {len(df_generado):,}")
print(f"Columnas: {df_generado.shape[1]}")
print(f"Tamaño del CSV en disco: {file_size_mb:,.2f} MB")
print(f"Tiempo de escritura: {t1 - t0:,.2f} s")

df_generado.head()

### Observación

El archivo en disco no parece enorme, pero al cargarlo en memoria puede ocupar **mucho más**.  
Eso pasa porque en memoria no solo guardamos texto: también hay índices, estructuras internas y objetos Python.

Vamos a comprobarlo.

## 2. Experimento 1: carga ingenua con pandas

Aquí vamos a leer el archivo con `pd.read_csv()` **sin optimizar nada**.

Eso significa que pandas va a inferir tipos por sí solo.  
En muchos casos prácticos, eso deja varias columnas como `object`, lo cual suele gastar mucha memoria.

In [0]:
gc.collect()

rss_inicio = rss_mb()
t0 = timer()

df_naive = pd.read_csv(CSV_PATH)

t1 = timer()
rss_fin = rss_mb()

resultado_naive = {
    "escenario": "pandas_naive",
    "tiempo_s": round(t1 - t0, 3),
    "memoria_dataframe_mb": round(df_mem_mb(df_naive), 2),
    "rss_inicio": round(rss_inicio,2),
    "rss_fin": round(rss_fin,2),
    "delta_rss_mb": round(rss_fin - rss_inicio, 2),
}

print("Resultado:")
print(resultado_naive)
print()
print("Tipos de datos detectados por pandas:")
display(df_naive.dtypes)

### ¿Qué estamos viendo?

- El tiempo de lectura.
- La memoria del `DataFrame`.
- El cambio de memoria del proceso (`RSS`).
- Los tipos de datos inferidos por pandas.

Lo más importante aquí es fijarse en cuántas columnas quedaron como **`object`**.

## 3. Experimento 2: carga optimizada con `dtype`

Ahora vamos a leer el mismo archivo, pero ayudándole a pandas:

- `id_venta` como `int32`
- `cantidad` como `int8`
- valores monetarios como `float32`
- columnas repetitivas como `category`
- `fecha` como fecha real

La meta aquí no es “hacer magia”, sino usar tipos más razonables.

In [0]:
del df_naive
gc.collect()

dtype_map = {
    "id_venta": "int32",
    "ciudad": "category",
    "canal": "category",
    "estado": "category",
    "producto": "category",
    "segmento": "category",
    "cantidad": "int8",
    "precio_unitario": "float32",
    "descuento_pct": "float32",
    "monto": "float32"
}

rss_inicio = rss_mb()
t0 = timer()

df_opt = pd.read_csv(
    CSV_PATH,
    dtype=dtype_map,
    parse_dates=["fecha"]
)

t1 = timer()
rss_fin = rss_mb()

resultado_opt = {
    "escenario": "pandas_optimizado",
    "tiempo_s": round(t1 - t0, 3),
    "memoria_dataframe_mb": round(df_mem_mb(df_opt), 2),
    "rss_inicio": round(rss_inicio,2),
    "rss_fin": round(rss_fin,2),
    "delta_rss_mb": round(rss_fin - rss_inicio, 2),
}

print("Resultado:")
print(resultado_opt)
print()
print("Tipos de datos optimizados:")
display(df_opt.dtypes)

In [0]:
comparacion_carga = pd.DataFrame([resultado_naive, resultado_opt])
comparacion_carga.head()

In [0]:
reduccion_memoria =round(((829.8-57.22)/829.80)*100,2)
print("% Porcentaje de reducción de memoria: ",reduccion_memoria, "%")

### Interpretación

Aquí suele ocurrir algo muy importante:

- a veces la lectura optimizada puede tardar parecido o incluso un poco más,
- pero la memoria baja.

Eso es una excelente noticia, porque en problemas reales la memoria suele ser el cuello de botella más peligroso.

## 4. Experimento 3: tipos categóricos

Ahora vamos a aislar el efecto de las columnas categóricas.

Primero cargamos algunas columnas como texto normal (`object`) y luego las convertimos a `category`.  
Así podemos ver el ahorro **columna por columna**.

In [0]:
df_texto = pd.read_csv(CSV_PATH, usecols=["ciudad", "canal", "estado", "producto", "segmento"]) # Sin definir tipos de datos categóricos 

mem_obj = (df_texto.memory_usage(deep=True) / 1048576).round(2) # Calcular memoria por columna 

df_cat = df_texto.astype("category") #Definir tipos de datos utilizando category

mem_cat = (df_cat.memory_usage(deep=True) / 1048576).round(2) # Calcular memoria por columna

comparacion_categorias = pd.DataFrame({
    "memoria_object_mb": mem_obj,
    "memoria_category_mb": mem_cat,
    "ahorro_mb": (mem_obj - mem_cat).round(2)
})

comparacion_categorias['Reducción de memoria (%)'] = (comparacion_categorias['ahorro_mb'] / comparacion_categorias['memoria_object_mb'] * 100).round(2)

comparacion_categorias.head(10)


In [0]:
tabla_grafico = comparacion_categorias.drop(index="Index").copy()

ax = tabla_grafico[["memoria_object_mb", "memoria_category_mb"]].plot(
    kind="bar",
    figsize=(10, 5),
    title="Memoria por columna: object vs category"
)
ax.set_ylabel("MB")
ax.set_xlabel("Columna")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### ¿Por qué `category` ahorra memoria?

Porque en vez de guardar el texto completo en cada fila, pandas guarda:

1. una lista pequeña de categorías únicas
2. un código interno para cada fila

Eso funciona muy bien cuando los valores se repiten mucho, por ejemplo:

- ciudad
- estado
- canal
- segmento

No suele ayudar tanto en columnas casi únicas, como un correo electrónico o un identificador libre.

## 5. Experimento 4: chunking

**Chunking** significa leer el archivo por partes.

En vez de cargar todo el CSV de una sola vez, lo leemos en bloques de, por ejemplo, 200.000 filas.

Vamos a resolver una tarea real:

> calcular el monto total aprobado por ciudad

Y la haremos de dos maneras:

- lectura completa
- lectura por chunks

In [0]:
def total_aprobado_por_ciudad(df):
    filtrado = df[df["estado"] == "APROBADA"]
    return filtrado.groupby("ciudad", observed=True)["monto"].sum().sort_values(ascending=False)

# Versión 1: leer de una vez solo las columnas necesarias
gc.collect()
rss_inicio = rss_mb()
t0 = timer()

df_small = pd.read_csv(
    CSV_PATH,
    usecols=["ciudad", "estado", "monto"],
    dtype={"ciudad": "category", "estado": "category", "monto": "float32"}
)

resultado_full = total_aprobado_por_ciudad(df_small)

t1 = timer()
rss_fin = rss_mb()

exp_full = {
    "escenario": "Lectura completa",
    "tiempo_s": round(t1 - t0, 3),
    "memoria_dataframe_mb": round(df_mem_mb(df_small), 2),
    "delta_rss_mb": round(rss_fin - rss_inicio, 2),
}

print(exp_full)
display(resultado_full.head())

In [0]:
# Versión 2: chunking
gc.collect()
rss_inicio = rss_mb()
peak_rss = rss_inicio
t0 = timer()

parciales = []

for chunk in pd.read_csv(
    CSV_PATH,
    usecols=["ciudad", "estado", "monto"],
    dtype={"ciudad": "category", "estado": "category", "monto": "float32"},
    chunksize=200000
):
    parciales.append(total_aprobado_por_ciudad(chunk))
    peak_rss = max(peak_rss, rss_mb())

resultado_chunk = (
    pd.concat(parciales)
    .groupby(level=0)
    .sum()
)

t1 = timer()

exp_chunk = {
    "escenario": "chunking_200k",
    "tiempo_s": round(t1 - t0, 3),
    "pico_rss_aprox_mb": round(peak_rss - rss_inicio, 2),
}

print(exp_chunk)
display(resultado_chunk.head())

print("¿Los resultados coinciden?:", resultado_full.round(2).equals(resultado_chunk.round(2)))

In [0]:
comparacion_chunking = pd.DataFrame([
    {
        "escenario": exp_full["escenario"],
        "tiempo_s": exp_full["tiempo_s"],
        "memoria_o_pico_mb": exp_full["memoria_dataframe_mb"],
        "nota": "memoria del DataFrame completo"
    },
    {
        "escenario": exp_chunk["escenario"],
        "tiempo_s": exp_chunk["tiempo_s"],
        "memoria_o_pico_mb": exp_chunk["pico_rss_aprox_mb"],
        "nota": "pico aproximado de RSS durante chunks"
    },
])

comparacion_chunking

### Interpretación de chunking

Chunking es muy útil cuando:

- el archivo no cabe completo en RAM,
- solo necesitas agregaciones o filtros,
- puedes combinar resultados parciales al final.

Chunking no es muy útil cuando necesitas:

- ordenar todo el dataset de forma global,
- joins complejos entre partes,
- ventanas o deduplicaciones globales.

En este ejemplo, chunking resuelve bien el problema porque la tarea se puede partir y luego recombinar.

## 6. Experimento 5: paralelización

Ahora vamos a comparar:

- una versión **secuencial**
- una versión **paralela** con `joblib`

### Importante

Aquí vamos a ver una lección muy valiosa:

> **paralelizar no garantiza ser más rápido**

Si la tarea es pequeña o sencilla, el costo de repartir trabajo entre procesos puede ser mayor que el beneficio.

In [0]:
# Usaremos el DataFrame pequeño ya cargado: df_small
# Lo dividimos en 4 partes aproximadamente iguales.
partes = np.array_split(df_small, 4)

def procesar_parte(parte):
    return total_aprobado_por_ciudad(parte)

# Secuencial
t0 = timer()
res_seq = pd.concat([procesar_parte(parte) for parte in partes]).groupby('ciudad').sum().sort_values(ascending=False) # Proceso secuencial y en orden
t1 = timer()

exp_seq = {
    "escenario": "secuencial_particionado",
    "tiempo_s": round(t1 - t0, 4),
}

print(exp_seq)


In [0]:
# Paralelo
t0 = timer()
res_par = Parallel(n_jobs=4)(delayed(procesar_parte)(parte) for parte in partes)
res_par = pd.concat(res_par).groupby('ciudad').sum().sort_values(ascending=False)
t1 = timer()

exp_par = {
    "escenario": "paralelo_joblib_4",
    "tiempo_s": round(t1 - t0, 4),
}

print(exp_par)

In [0]:
comparacion_paralelo = pd.DataFrame([exp_seq, exp_par])
comparacion_paralelo["mas_rapido_que_secuencial"] = comparacion_paralelo["tiempo_s"] < comparacion_paralelo.loc[0, "tiempo_s"]

display(comparacion_paralelo)
print("¿Los resultados coinciden?:", res_seq.round(2).equals(res_par.round(2)))

### ¿Qué pasó?

Si la versión paralela sale más lenta, **no es un error**.  
De hecho, es una lección muy realista.

¿Por qué puede pasar?

- crear procesos tiene costo,
- mover datos entre procesos tiene costo,
- combinar resultados tiene costo,
- la tarea misma quizá era demasiado simple.

Por eso, en pandas muchas veces primero ganas más optimizando:

- `usecols`
- `dtype`
- `category`
- vectorización
- chunking

Y solo después evalúas si paralelizar realmente vale la pena.

## 7. Resumen general de resultados

Vamos a reunir los resultados principales en una sola tabla.

In [0]:
resumen = pd.DataFrame([
    {
        "escenario": resultado_naive["escenario"],
        "tiempo_s": resultado_naive["tiempo_s"],
        "memoria_mb": resultado_naive["memoria_dataframe_mb"],
        "comentario": "Carga sin optimizar"
    },
    {
        "escenario": resultado_opt["escenario"],
        "tiempo_s": resultado_opt["tiempo_s"],
        "memoria_mb": resultado_opt["memoria_dataframe_mb"],
        "comentario": "Carga con dtypes + category"
    },
    {
        "escenario": exp_full["escenario"],
        "tiempo_s": exp_full["tiempo_s"],
        "memoria_mb": exp_full["memoria_dataframe_mb"],
        "comentario": "Columnas necesarias"
    },
    {
        "escenario": exp_chunk["escenario"],
        "tiempo_s": exp_chunk["tiempo_s"],
        "memoria_mb": exp_chunk["pico_rss_aprox_mb"],
        "comentario": "Procesamiento por chunks"
    },
    {
        "escenario": exp_seq["escenario"],
        "tiempo_s": exp_seq["tiempo_s"],
        "memoria_mb": np.nan,
        "comentario": "Partes procesadas en secuencia"
    },
    {
        "escenario": exp_par["escenario"],
        "tiempo_s": exp_par["tiempo_s"],
        "memoria_mb": np.nan,
        "comentario": "Partes procesadas en paralelo"
    },
])

resumen

## 8. Conclusiones explicadas de forma simple

### 1. Pandas funciona muy bien si el problema cabe en memoria
Para muchas tareas de análisis, limpieza y agregación, pandas es suficiente y muy cómodo.

### 2. Optimizar tipos puede cambiarlo todo
El mismo archivo puede ocupar muchísimo menos en RAM si defines bien los `dtype`.

### 3. `category` es una gran herramienta para texto repetido
Si una columna tiene pocos valores únicos repetidos muchas veces, `category` puede ahorrar muchísimo.

### 4. Chunking sirve para “respirar” cuando el archivo es grande
No siempre es el camino más cómodo, pero sí es muy útil cuando no puedes cargar todo de una vez.

### 5. Paralelizar no siempre mejora
Ese es uno de los aprendizajes más importantes de este laboratorio.

Si la tarea es pequeña o el costo de coordinar procesos es alto, la versión paralela puede ser peor.

## 9. Reglas prácticas

Usa esta guía rápida:

- **Si el archivo cabe en memoria:** empieza con pandas normal
- **Si hay muchas columnas que no necesitas:** usa `usecols`
- **Si ves muchas columnas `object`:** revisa si deben ser `category`, `int32`, `float32`, etc.
- **Si el archivo es demasiado grande:** usa `chunksize`
- **Si estás pensando en paralelizar:** primero mide; no asumas que será mejor

La palabra clave aquí es **medir**.  
No adivines: prueba, compara y decide.

## 10. Ejercicios sugeridos

1. Cambia el `chunksize` a `50_000`, `100_000` y `500_000`.  
   ¿Cuál te da mejor equilibrio entre tiempo y memoria?

2. Quita `category` de una o dos columnas y vuelve a medir.  
   ¿Cuánto cambia la memoria?

3. Agrega una columna casi única, por ejemplo un `email` o `codigo_unico`.  
   ¿Sigue valiendo la pena usar `category`?

4. Cambia la tarea de negocio: en vez de sumar `monto`, cuenta ventas por `producto` y `canal`.


## 11. Cierre

Si te quedas con una sola idea de este notebook, que sea esta:

> **Antes de paralelizar pandas, optimiza memoria y simplifica el trabajo.**

Esa decisión suele dar mejores resultados que intentar “forzar velocidad” desde el principio.